# COCO Ingestion: Hugging Face → RustFS → DuckLake

This notebook ingests a small real COCO object-detection subset from the Hugging Face Hub.

- Heavy image bytes → RustFS (`s3://lakehouse/assets/coco/...`)
- Image metadata + annotations → DuckLake raw tables
- No pixel/image blobs are stored in DuckLake tables

Dataset used: `detection-datasets/coco`, validation split, streaming mode.


In [1]:
import io
import boto3
import duckdb
import pandas as pd

from datasets import load_dataset
from PIL import Image

BUCKET = "lakehouse"
S3_ENDPOINT = "http://rustfs:9000"
N_IMAGES = 1000


## 1. Load a small real COCO subset from Hugging Face

Streaming avoids downloading the full dataset.


In [2]:
ds = load_dataset(
    "detection-datasets/coco",
    split="val",
    streaming=True,
)

print(ds)


Resolving data files:   0%|          | 0/40 [00:00<?, ?it/s]

IterableDataset({
    features: ['image_id', 'image', 'width', 'height', 'objects'],
    num_shards: 2
})


## 2. Connect to RustFS


In [3]:
s3 = boto3.client(
    "s3",
    endpoint_url=S3_ENDPOINT,
    aws_access_key_id="rustfsadmin",
    aws_secret_access_key="rustfsadmin",
)

print([b["Name"] for b in s3.list_buckets()["Buckets"]])


['lakehouse']


## 3. Extract images and annotations


In [4]:
image_rows = []
annotation_rows = []

category_names = None
try:
    category_feature = ds.features["objects"]["category"].feature
    category_names = category_feature.names
except Exception:
    pass

for i, example in enumerate(ds):
    if i >= N_IMAGES:
        break

    image_id = int(example["image_id"])
    image = example["image"].convert("RGB")

    buffer = io.BytesIO()
    image.save(buffer, format="JPEG", quality=95)
    image_bytes = buffer.getvalue()

    key = f"assets/coco/{image_id}.jpg"

    s3.put_object(
        Bucket=BUCKET,
        Key=key,
        Body=image_bytes,
        ContentType="image/jpeg",
    )

    image_uri = f"s3://{BUCKET}/{key}"

    image_rows.append({
        "image_id": image_id,
        "image_uri": image_uri,
        "width": int(example["width"]),
        "height": int(example["height"]),
        "split": "val",
        "source_dataset": "detection-datasets/coco",
    })

    objects = example["objects"]
    bbox_ids = objects.get("bbox_id", [])
    categories = objects.get("category", [])
    bboxes = objects.get("bbox", [])
    areas = objects.get("area", [])

    for j, bbox in enumerate(bboxes):
        category_id = int(categories[j])
        if category_names and 0 <= category_id < len(category_names):
            category = category_names[category_id]
        else:
            category = str(category_id)

        annotation_rows.append({
            "image_id": image_id,
            "image_uri": image_uri,
            "bbox_id": int(bbox_ids[j]) if j < len(bbox_ids) else None,
            "category_id": category_id,
            "category": category,
            "bbox_xmin": float(bbox[0]),
            "bbox_ymin": float(bbox[1]),
            "bbox_xmax": float(bbox[2]),
            "bbox_ymax": float(bbox[3]),
            "area": float(areas[j]) if j < len(areas) else None,
            "split": "val",
        })

print(f"Uploaded {len(image_rows)} COCO images to RustFS")
print(f"Prepared {len(annotation_rows)} annotation rows")


Uploaded 1000 COCO images to RustFS
Prepared 7266 annotation rows


## 4. Inspect extracted metadata


In [5]:
images_df = pd.DataFrame(image_rows)
annotations_df = pd.DataFrame(annotation_rows)

display(images_df.head())
display(annotations_df.head())


,image_id,image_uri,width,height,split,source_dataset
0,139,s3://lakehouse/assets/coco/139.jpg,640,426,val,detection-datasets/coco
1,285,s3://lakehouse/assets/coco/285.jpg,586,640,val,detection-datasets/coco
2,632,s3://lakehouse/assets/coco/632.jpg,640,483,val,detection-datasets/coco
3,724,s3://lakehouse/assets/coco/724.jpg,375,500,val,detection-datasets/coco
4,776,s3://lakehouse/assets/coco/776.jpg,428,640,val,detection-datasets/coco


,image_id,image_uri,bbox_id,category_id,category,bbox_xmin,bbox_ymin,bbox_xmax,bbox_ymax,area,split
0,139,s3://lakehouse/assets/coco/139.jpg,26547,58,potted plant,236.98,142.51,261.68,212.01,531.80710,val
1,139,s3://lakehouse/assets/coco/139.jpg,34646,62,tv,7.03,167.76,156.35,262.63,13244.65770,val
2,139,s3://lakehouse/assets/coco/139.jpg,35802,62,tv,557.21,209.19,638.56,287.92,5833.11795,val
3,139,s3://lakehouse/assets/coco/139.jpg,103487,56,chair,358.98,218.05,414.98,320.88,2245.34355,val
4,139,s3://lakehouse/assets/coco/139.jpg,104368,56,chair,290.69,218.00,352.52,316.48,1833.78400,val


## 5. Stage metadata locally as Parquet


In [6]:
images_df.to_parquet("/data/local/coco_images_raw.parquet", index=False)
annotations_df.to_parquet("/data/local/coco_annotations_raw.parquet", index=False)
print("Staged COCO image metadata and annotations as Parquet")


Staged COCO image metadata and annotations as Parquet


## 6. Attach DuckLake


In [7]:
con = duckdb.connect()
con.execute(open("../sql/00_attach.sql").read())
print("DuckLake attached")


DuckLake attached


## 7. Create raw COCO tables


In [8]:
con.execute("""
CREATE OR REPLACE TABLE raw.coco_images AS
SELECT *
FROM read_parquet('/data/local/coco_images_raw.parquet');
""")

con.execute("""
CREATE OR REPLACE TABLE raw.coco_annotations AS
SELECT *
FROM read_parquet('/data/local/coco_annotations_raw.parquet');
""")

print("Created raw.coco_images and raw.coco_annotations")


Created raw.coco_images and raw.coco_annotations


## 8. Verify the raw layer


In [9]:
print("Images:")
display(con.sql("""
    SELECT image_id, image_uri, width, height, split
    FROM raw.coco_images
    ORDER BY image_id
""").df())

print("Annotations:")
display(con.sql("""
    SELECT image_id, category, bbox_xmin, bbox_ymin, bbox_xmax, bbox_ymax
    FROM raw.coco_annotations
    LIMIT 10
""").df())


Images:


,image_id,image_uri,width,height,split
0,139,s3://lakehouse/assets/coco/139.jpg,640,426,val
1,285,s3://lakehouse/assets/coco/285.jpg,586,640,val
2,632,s3://lakehouse/assets/coco/632.jpg,640,483,val
3,724,s3://lakehouse/assets/coco/724.jpg,375,500,val
4,776,s3://lakehouse/assets/coco/776.jpg,428,640,val
...,...,...,...,...,...
995,119516,s3://lakehouse/assets/coco/119516.jpg,640,427,val
996,119641,s3://lakehouse/assets/coco/119641.jpg,640,480,val
997,119677,s3://lakehouse/assets/coco/119677.jpg,640,480,val
998,119828,s3://lakehouse/assets/coco/119828.jpg,500,375,val


Annotations:


,image_id,category,bbox_xmin,bbox_ymin,bbox_xmax,bbox_ymax
0,139,potted plant,236.98,142.51,261.68,212.01
1,139,tv,7.03,167.76,156.35,262.63
2,139,tv,557.21,209.19,638.56,287.92
3,139,chair,358.98,218.05,414.98,320.88
4,139,chair,290.69,218.00,352.52,316.48
5,139,chair,413.20,223.01,443.37,304.37
6,139,chair,317.40,219.24,338.98,230.83
7,139,person,412.80,157.61,465.85,295.62
8,139,person,384.43,172.21,399.55,207.95
9,139,microwave,512.22,205.75,526.96,221.72


## 9. COCO metadata query

Find images containing at least 5 instances of the same category.


In [10]:
con.sql("""
SELECT
    image_uri,
    category,
    COUNT(*) AS n_objects
FROM raw.coco_annotations
GROUP BY image_uri, category
HAVING COUNT(*) >= 5
ORDER BY n_objects DESC
""")


┌───────────────────────────────────────┬──────────┬───────────┐
│               image_uri               │ category │ n_objects │
│                varchar                │ varchar  │   int64   │
├───────────────────────────────────────┼──────────┼───────────┤
│ s3://lakehouse/assets/coco/103548.jpg │ sheep    │        19 │
│ s3://lakehouse/assets/coco/87470.jpg  │ cow      │        15 │
│ s3://lakehouse/assets/coco/2299.jpg   │ person   │        14 │
│ s3://lakehouse/assets/coco/84270.jpg  │ person   │        14 │
│ s3://lakehouse/assets/coco/79969.jpg  │ person   │        14 │
│ s3://lakehouse/assets/coco/97585.jpg  │ vase     │        14 │
│ s3://lakehouse/assets/coco/24021.jpg  │ person   │        14 │
│ s3://lakehouse/assets/coco/26690.jpg  │ person   │        14 │
│ s3://lakehouse/assets/coco/78565.jpg  │ person   │        14 │
│ s3://lakehouse/assets/coco/31322.jpg  │ bird     │        14 │
│                  ·                    │  ·       │         · │
│                  ·     

## 10. Verify DuckLake snapshots


In [11]:
con.sql("FROM lake.snapshots()")


┌─────────────┬───────────────────────────────┬────────────────┬─────────────────────────────────────────────────────────────────────────────────────────┬─────────┬────────────────┬───────────────────┐
│ snapshot_id │         snapshot_time         │ schema_version │                                         changes                                         │ author  │ commit_message │ commit_extra_info │
│    int64    │   timestamp with time zone    │     int64      │                                 map(varchar, varchar[])                                 │ varchar │    varchar     │      varchar      │
├─────────────┼───────────────────────────────┼────────────────┼─────────────────────────────────────────────────────────────────────────────────────────┼─────────┼────────────────┼───────────────────┤
│           0 │ 2026-08-10 03:23:52.987195+00 │              0 │ {schemas_created=[main]}                                                                │ NULL    │ NULL           │ NULL      

In [12]:
print("images_df:", len(images_df))
print("annotations_df:", len(annotations_df))

print(
    "raw.coco_images:",
    con.sql("SELECT COUNT(*) FROM raw.coco_images").fetchone()[0]
)

print(
    "raw.coco_annotations:",
    con.sql("SELECT COUNT(*) FROM raw.coco_annotations").fetchone()[0]
)

images_df: 1000
annotations_df: 7266
raw.coco_images: 1000
raw.coco_annotations: 7266
